**1)**
Set up the parameters

In [ ]:
# 这个DDM调参实验中，我探索了初始的高斯分布的方差以及DDM模型漂移系数对结果的影响。
# 初始方差越大，越难收敛，甚至还可能出现相反的结果。
# 而漂移系数越大，收敛越快。

In [1]:
using Distributions
using PlotlyJS
using Random

# --- DDM Parameters ---
k = 0.3
σ = 1.0   # Noise standard deviation
B = 2.0   # Decision boundary
dt = 0.001 # Time step
max_t = 5.0 # Maximum simulation time
max_t_steps = Int(max_t/dt)


┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24


5000

**2)**
Here is the main function for simulating DDM

In [2]:
# --- Simulation Function ---
function simulate_ddm(k, σ, B, dt, max_t)
    x = 0.0  # Initial decision variable
    xs = zeros(max_t_steps)
    r = 0    # undetermined
    t_step = 1

    while abs(x) < B && t_step < max_t_steps
        dx = k * dt + σ * randn() * sqrt(dt) # Euler-Maruyama integration
        x += dx
        xs[t_step] = x
        t_step = t_step + 1
    end
    if x>=B
        r=1
        xs[t_step:max_t_steps] .= B
    elseif x<=-B
        r=-1
        xs[t_step:max_t_steps] .= -B
    end
    return t_step * dt, xs, r # 1 for upper boundary, -1 for lower
end


simulate_ddm (generic function with 1 method)

**3)**
Let's run a bunch of simulations, recording the reaction times and the average drifting trajectory.

In [3]:
# --- Run Multiple Trials ---
n_trials = 10000
decisions = zeros(Int, n_trials)
rts_correct = []
rts_error = []
n_correct = n_error = 0
trace_correct = zeros(max_t_steps)
trace_error = zeros(max_t_steps)

for i in 1:n_trials
    rt, xs, decision = simulate_ddm(k, σ, B, dt, max_t)
    decisions[i] = decision
    # Record reaction time
    if decision == 1 
        push!(rts_correct, rt) 
        n_correct = n_correct + 1
        trace_correct = ((n_correct-1) * trace_correct + xs) / n_correct
    elseif decision == -1
        push!(rts_error, rt)
        n_error = n_error + 1
        trace_error = ((n_error-1) * trace_error + xs) / n_error
    end
end


**4)**
Generating the plots

In [4]:
# Reaction Time Histogram
histogram_correct = histogram(x=rts_correct, nbinsx=20, name="Correct", opacity=0.6)
histogram_error = histogram(x=rts_error, nbinsx=20, name="Error", opacity=0.6)

layout_hist = Layout(title="Reaction Time Distribution", xaxis_title="Time (s)", yaxis_title="Frequency", bar_mode="overlay")
display(plot([histogram_correct, histogram_error], layout_hist))

# Average Drifting Trajectories for the 2 choices
trajectory_correct = scatter(x=1:max_t_steps, y=trace_correct, mode="lines", name="Correct Trials")
trajectory_error = scatter(x=1:max_t_steps, y=trace_error, mode="lines", name="Error Trials")
upper_bound = scatter(x=[1, max_t_steps], y=[B,B], mode="lines", name="Upper Bound", line=attr(dash="dash"))
lower_bound = scatter(x=[1, max_t_steps], y=[-B,-B], mode="lines", name="Lower Bound", line=attr(dash="dash"))

layout_traj = Layout(title="Average DDM Trajectory", xaxis_title="Time (s)", yaxis_title="x")

display(plot([trajectory_correct, trajectory_error, upper_bound, lower_bound], layout_traj))


# --- Analysis (Example: Accuracy and Mean RT) ---
accuracy = n_correct / (n_correct + n_error) # Assuming '1' is the correct choice

println("Accuracy: ", accuracy)
println("Mean RT (correct): ", mean(rts_correct))
println("Mean RT (error): ", mean(rts_error))


┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "histogram with fields name, nbinsx, opacity, type, and x",
  "histogram with fields name, nbinsx, opacity, type, and x"
]

layout: "layout with fields bar, margin, template, title, xaxis, and yaxis"

data: [
  "scatter with fields mode, name, type, x, and y",
  "scatter with fields mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"

Accuracy: 0.7598064853556485
Mean RT (correct): 2.3727478919291
Mean RT (error): 2.3273527490473596


**5)**
Now let's use DDM to simulate the random dots experiment. We will include 6 coherence levels, test the model accuracies and reaction times

In [9]:
using Random

# Parameters
B = 3
# drift_rate = 6
# drift_rate = 20
drift_rate = 0.5
σ = 2
# σ = 5
time_non_decision = 0.3
coherences = [-0.512, -.256, -.128, -.064, -.032, 0, +.032, +.064, +.128, +.256, +.512]
n_coherences = length(coherences)
total_trials = 10000
dt = 0.01
max_t = 2.0

# Simulating original DDM
function simulate_ddm(drift, σ, B, dt, max_t)
    accum = 0
    rt = 0
    t = 0
    while t < max_t
        accum += drift * dt + σ * sqrt(dt) * randn()
        t += dt
        rt += dt
        if abs(accum) >= B
            decision = sign(accum)
            return rt, accum, decision
        end
    end
    return rt, accum, 0  # No decision
end

# Initialize arrays
choices_original = [[] for _ in 1:n_coherences]
rts_original = [[] for _ in 1:n_coherences]
choices_collapsing = [[] for _ in 1:n_coherences]
rts_collapsing = [[] for _ in 1:n_coherences]
choices_leaky = [[] for _ in 1:n_coherences]
rts_leaky = [[] for _ in 1:n_coherences]

# Run simulations for original DDM
for trial in 1:total_trials
    coh_i = rand(1:n_coherences)
    coh = coherences[coh_i]
    rt, _, decision = simulate_ddm(drift_rate * coh, σ, B, dt, max_t)
    if decision != 0
        push!(choices_original[coh_i], decision / 2 + 0.5)  # Convert decision into 0 and 1
        push!(rts_original[coh_i], rt + time_non_decision)
    end
end


# Calculate probabilities and reaction times for original DDM
p_right = []
rt = []
for coh_i in 1:n_coherences
    if length(choices_original[coh_i]) > 0
        push!(p_right, mean(choices_original[coh_i]))
        push!(rt, mean(rts_original[coh_i]))
    else
        push!(p_right, -1)
        push!(rt, 0)
    end
end

# Output results
println("Original DDM Results:")
println("p_right: ", p_right)
println("rt: ", rt)


Original DDM Results:
p_right: Any[0.3651452282157676, 0.483739837398374, 0.4847161572052402, 0.49390243902439024, 0.5072164948453608, 0.48927875243664715, 0.4934579439252336, 0.5522682445759369, 0.5297504798464492, 0.5536480686695279, 0.6177062374245473]
rt: Any[1.4079668049792546, 1.4279268292682925, 1.4410043668122265, 1.4152642276422778, 1.4445567010309293, 1.422241715399612, 1.437439252336449, 1.4415384615384608, 1.442936660268715, 1.3924678111588003, 1.3908048289738433]


In [10]:
using PlotlyJS

# Plot results using PlotlyJS

# 原始 DDM 结果
trace1_original = scatter(x=coherences, y=p_right, mode="lines+markers", name="Original DDM")
layout1_original = Layout(title="Psychometric Curve(Original)", xaxis_title="Coherences", yaxis_title="Right Choices")
p1_original = plot([trace1_original], layout1_original)

trace2_original = scatter(x=coherences, y=rt, mode="lines+markers", name="Original DDM")
layout2_original = Layout(title="Chronometric Curve(Original)", xaxis_title="Coherences", yaxis_title="RT")
p2_original = plot([trace2_original], layout2_original)

# 合并图像
display(p1_original)

display(p2_original)

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields mode, name, type, x, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"

data: [
  "scatter with fields mode, name, type, x, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"